In [1]:
import pandas as pd
import numpy as np
import genesis as gs

import torch
from rsl_rl.algorithms import PPO
# from rsl_rl.modules.actor_critic import ActorCritic
from rsl_rl.runners import OnPolicyRunner
from rsl_rl.runners import OnPolicyRunner

from tensordict import TensorDict
from environment import Environment

gs.init(backend=gs.gpu)

[I 11/01/26 17:59:51.769 17913] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


[Genesis] [18:00:00] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [18:00:00] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [18:00:00] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [18:00:00] [INFO] Running on [NVIDIA GeForce RTX 3070] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [18:00:00] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO


# モデリング少しやり方検討

In [9]:
# RLの部分
    # import genesis as gs
# env = CustomObstacleEnv(num_envs=2048, device="cuda")

class RLLearner:
    def __init__(self, env):
        self.env = env
        self.num_envs = env.num_envs
        self.num_obs = env.num_obs
        self.num_privileged_obs = None
        self.num_actions = env.num_actions
        self.device = env.device
        self.cfg = {"env": self.env, 
                    "num_envs": self.num_envs, 
                    "num_obs":self.num_obs, 
                    "num_actions": self.num_actions}  # 追加

    def _to_tensordict(self, obs):
        return TensorDict(
            {"policy": obs},
            batch_size=[self.num_envs],
            device=self.device,
        )

    def get_observations(self):
        obs = self.env._get_obs()
        return self._to_tensordict(obs)

    def step(self, actions):
        obs, rewards, dones = self.env.steps(actions)
        obs = self._to_tensordict(obs)
        return obs, rewards, dones, {}

    def reset(self):
        return self.env.reset(), {}


def train(FILE_NAME="20261101_rl_learning_baseline.mp4", num_envs=15):
    # gs.init(backend=gs.gpu)

    raw_env = Environment(num_env=num_envs, device="cuda")
    env = RLLearner(raw_env)

    train_cfg = {
    # Runner settings
    "num_steps_per_env": 150,
    "save_interval": 100,

    "obs_groups": {
        "actor": ["policy"],
        "critic": ["policy"],
    },

    # PPO settings
    "algorithm": {
        "class_name": "PPO",
        "learning_rate": 1e-3,
        "num_learning_epochs": 5,
        "num_mini_batches": 4,
        "clip_param": 0.2,
        "desired_kl": 0.01,
        "gamma": 0.99,
        "lam": 0.95,
        "entropy_coef": 0.01,
        "value_loss_coef": 1.0,
        "max_grad_norm": 1.0,
    },

    # Actor
    "actor": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
        "distribution_cfg": {
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
        },
    },

    # Critic
    "critic": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
    },}
    ppo_runner = OnPolicyRunner(
        env=env,
        train_cfg=train_cfg,
        log_dir="logs/go2_rl",
        device=env.device,
    )

    env.env.cam.start_recording(
        FILE_NAME,
        fps=30,
    )
    print("camera-started")

    ppo_runner.learn(
        num_learning_iterations=100,
        # init_at_random_policy=False,
    )

    env.env.cam.stop_recording()

    return env, ppo_runner


In [ ]:
FILE_NAME = "20261102_rl_learning_baseline.mp4"
ENVS_NUM = 5 #自分の環境じゃ無理


In [11]:
models = train(FILE_NAME, ENVS_NUM)

[Genesis] [18:50:06] [INFO] Scene <bb06596> created.
[Genesis] [18:50:06] [INFO] Adding <gs.engine.entities.RigidEntity> plane_4d0ddfb. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [18:50:08] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_7f2ac05. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.12/site-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [18:50:08] [INFO] Building scene <bb06596>...
[Genesis] [18:50:08] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [18:50:09] [INFO] Compiling simulation kernels...
[Genesis] [18:50:09] [INFO] Building visualizer...
[Genesis] [18:50:09] [INFO] Software rendering context detected. Shadows and plane reflection not supported. Beware rendering will be extremely slow.
[Genesis] [18:50:09] [INFO] Raytracer and Rasterizer requires binding to the camera with a specific environment index. Defaulting to 'rendered_envs_idx[0

In [16]:
# # 推論用にモデルを保存
# ppo_runner = models[1]
# ppo_runner.save("final_model.pt")

In [12]:
from enviroment_penalty import EnvironmentPenalty

In [19]:
def train_penalty(FILE_NAME="20261101_rl_learning_baseline.mp4", num_envs=15):
    # gs.init(backend=gs.gpu)

    raw_env = EnvironmentPenalty(num_env=num_envs, device="cuda")
    env = RLLearner(raw_env)

    train_cfg = {
    # Runner settings
    "num_steps_per_env": 150,
    "save_interval": 100,

    "obs_groups": {
        "actor": ["policy"],
        "critic": ["policy"],
    },

    # PPO settings
    "algorithm": {
        "class_name": "PPO",
        "learning_rate": 1e-3,
        "num_learning_epochs": 5,
        "num_mini_batches": 4,
        "clip_param": 0.2,
        "desired_kl": 0.01,
        "gamma": 0.99,
        "lam": 0.95,
        "entropy_coef": 0.01,
        "value_loss_coef": 1.0,
        "max_grad_norm": 1.0,
    },

    # Actor
    "actor": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
        "distribution_cfg": {
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
        },
    },

    # Critic
    "critic": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
    },}
    ppo_runner = OnPolicyRunner(
        env=env,
        train_cfg=train_cfg,
        log_dir="logs/go2_rl",
        device=env.device,
    )

    env.env.cam.start_recording(
        FILE_NAME,
        fps=30,
    )
    print("camera-started")

    ppo_runner.learn(
        num_learning_iterations=100,
        # init_at_random_policy=False,
    )

    env.env.cam.stop_recording()

    return env, ppo_runner

In [20]:
FILE_NAME = "20261102_rl_learning_with_penalty.mp4"
ENVS_NUM = 5 #自分の環境じゃ無理
models = train_penalty(FILE_NAME, ENVS_NUM)

[Genesis] [20:02:47] [INFO] Scene <3ca72e6> created.
[Genesis] [20:02:47] [INFO] Adding <gs.engine.entities.RigidEntity> plane_82b8b84. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [20:02:49] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_eec41dd. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.12/site-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [20:02:49] [INFO] Building scene <3ca72e6>...
[Genesis] [20:02:49] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [20:02:50] [INFO] Compiling simulation kernels...
[Genesis] [20:02:50] [INFO] Building visualizer...
[Genesis] [20:02:50] [INFO] Software rendering context detected. Shadows and plane reflection not supported. Beware rendering will be extremely slow.
[Genesis] [20:02:50] [INFO] Raytracer and Rasterizer requires binding to the camera with a specific environment index. Defaulting to 'rendered_envs_idx[0

In [22]:
penalty_model = models[1]
penalty_model.save("model_penalty.pt")

# 推論部分

In [ ]:
# 一旦

In [53]:
class EnvironmentInf: 
    def __init__(self, num_env=1, device="cuda"):
        self.num_envs = num_env
        self.device = device
        
    
        # シーン構築
        self.scene = gs.Scene(
            sim_options= gs.options.SimOptions(dt=0.01, substeps=2), # 本来は細かくした方がいいかもしれんが、メモリが足りない
            show_viewer=False,
        )

        self.plane = self.scene.add_entity(gs.morphs.Plane())
        self.robot = self.scene.add_entity(gs.morphs.URDF(
            file="urdf/go2/urdf/go2.urdf",
            pos=(0, 0, 0.5)))


        self.cam = self.scene.add_camera(
            res=(1280, 720),
            pos=(3.0, -3.0, 5), # カメラの配置位置 (x, y, z) 
            lookat=(0.0, 0.0, 0.5), # 注視点 (ロボットの少し上など) 
            fov=40, # 視野角 
            GUI=False, # GUIはオフ 
            )


        # 並列構築したい数を渡す
        self.scene.build(n_envs=self.num_envs)
        # ビルド後, 実態が生成
        self.initial_pos = self.robot.get_pos()
        self.initial_state = self.scene.get_state()
        # デフォルの関節角/PDゲインを設定
        self.n_dofs = self.robot.n_dofs
        self.default_dof_pos = torch.zeros((self.num_envs, self.n_dofs), device=self.device)
        self.robot.set_dofs_kp([20.0] * self.n_dofs)
        self.robot.set_dofs_kv([0.5] * self.n_dofs)

        # 観測・行動がもつ次元数の定義
        self.num_obs = self.n_dofs * 2 + 2
        self.num_actions = self.n_dofs # 関節角への指示　＝＝行動
        self.goal_pos = [20, 0, 0]


        # stepで学習やターゲットに使用したデータを格納する
        self.logs ={
            "target_pos": [],
            "rewards": [],
            "observation": [],
            "is_done": [] #終了判定
        }
        self.cfg = {
                    "num_envs": self.num_envs,
                    "device": self.device,
                    "default_position": self.default_dof_pos
                }
        return None

    def reset(self):
        self.scene.reset()
        return self._get_obs()
    
    def steps(self, actions):
        actions = actions.to(self.device)
        target_pos = self.default_dof_pos + actions
        self.robot.control_dofs_position(target_pos)
        
        
        # 物理時間を進める
        self.scene.step()

        # 観測/報酬決定
        obs = self._get_obs()
        rewards = self._compute_rewards()
        dones, fell = self._check_dones()

         # 転倒や目的を達成したepisodeは初期化
        # if (dones.numel() > 0):
        #     self._reset_falling_robot(fell) # 推論時は無効化

        # ここでlogging
        self._logging(target_pos, obs, rewards, dones)
        return obs, rewards, dones

    def _logging(self, target_pos, obs, rewards, dones):
        self.logs["target_pos"].append(target_pos)
        self.logs["observation"].append(obs)
        self.logs["rewards"].append(rewards)
        self.logs["is_done"].append(dones)
        return None

    def _reset_falling_robot(self,fall):#　転倒ロボの環境を初期化
        env_idx = torch.where(fall)[0] #転倒したロボット取得
        if len(env_idx) > 0:
            self.scene.reset(state = self.initial_state , envs_idx=env_idx)
        return None 

    def _get_obs(self): #現在のロボットの状態取得
        # ロボットの関節状態
        dofs_position = self.robot.get_dofs_position()
        dofs_velocity = self.robot.get_dofs_velocity()
        current_state = torch.cat([dofs_position, dofs_velocity], dim=-1).to(self.device)

        # イメージはこれ assert self.num_obj.shape == current_state.shape
        return current_state

    def _compute_rewards(self):
        """
        報酬：終了目標に近づく /　頭の位置を高くしている / goal していたら、bonus
        
        """
        robot_pos = self.robot.get_pos()
        # きめの問題だから、x方向に進んだかどうか？
        # 近づいたかを算出する
        reached_goal = robot_pos[:, 0] >= 20.0
        reward_goal = torch.where(reached_goal, 15, 0) # ゴールした場合のボーナス
        # 通常の報酬
        reward_fwd = self.robot.get_dofs_velocity()[:,0] * 2.0# 速度の方がいいらしい、腹落ち要件

        # +α: 転倒時に罰則を与えるわ => 逆転で頭をあげてられる＝＝姿勢を安定させる
        penelize_falling_idx = robot_pos[:, 2] > 0.35
        stable_posture = torch.where(penelize_falling_idx, 2, 0)
        return reward_fwd+reward_goal+stable_posture

    def _check_dones(self):
        # 全環境のロボット位置を取得 
        pos = self.robot.get_pos() 
        #1. X軸方向に20m到達・通過した判定 (Bool Tensor) 
        reached_goal = pos[:, 0] >= 20.0 
        # 2. 転倒判定（例: 胴体の高さが0.2m未満に低下） => 学習した結果が倒れた状態でばたばたになっている
        fell = pos[:, 2] < 0.35

        # todoこれはせt タイムアウト判定（例: 最大ステップ数を超過） #
        # timeout = self.episode_length_buf >= self.max_episode_length 
        # いずれかの条件を満たした環境を True (終了) にする 
        dones = reached_goal | fell 
        return dones, fell

In [56]:
import torch

train_cfg = {
    # Runner settings
    "num_steps_per_env": 150,
    "save_interval": 100,

    "obs_groups": {
        "actor": ["policy"],
        "critic": ["policy"],
    },

    # PPO settings
    "algorithm": {
        "class_name": "PPO",
        "learning_rate": 1e-3,
        "num_learning_epochs": 5,
        "num_mini_batches": 4,
        "clip_param": 0.2,
        "desired_kl": 0.01,
        "gamma": 0.99,
        "lam": 0.95,
        "entropy_coef": 0.01,
        "value_loss_coef": 1.0,
        "max_grad_norm": 1.0,
    },

    # Actor
    "actor": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
        "distribution_cfg": {
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
        },
    },

    # Critic
    "critic": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
    },}
# 更新しちゃったからこで取得
raw_env = EnvironmentInf(num_env=1, device="cuda")
env = RLLearner(raw_env)

ppo_runner = OnPolicyRunner(
    env=env,
    train_cfg=train_cfg,
    log_dir=None,
    device=env.device,
)

ppo_runner.load("final_model.pt")

policy = ppo_runner.get_inference_policy(device=env.device)
# 学習済みActor取得
policy = ppo_runner.get_inference_policy(device=env.device)

# 動画設定
duration_sec = 90
dt = 0.01
FPS_MP4 = 30
total_steps = int(FPS_MP4 * duration_sec *3 ) # 何故か, 90*30で30秒の動画にしかならん

# 環境リセット
env.reset()

# RSL-RL用のObservation取得
obs = env.get_observations()

# 録画開始
env.env.cam.start_recording(
    save_to_filename="20261102_robot_90sec_提出用.mp4",
    fps=FPS_MP4,
)

with torch.inference_mode():
    for step in range(total_steps):
        
        # Actorによる推論
        actions = policy(obs)

        # 環境を進める
        obs, rewards, dones, infos = env.step(actions)
        # print(obs)



env.env.cam.stop_recording()

print("録画完了: robot_90sec.mp4")

[Genesis] [21:01:08] [INFO] Scene <59dcb14> created.
[Genesis] [21:01:08] [INFO] Adding <gs.engine.entities.RigidEntity> plane_6ec7a83. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [21:01:10] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_8d2f0f5. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.12/site-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [21:01:10] [INFO] Building scene <59dcb14>...
[Genesis] [21:01:10] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [21:01:11] [INFO] Compiling simulation kernels...
[Genesis] [21:01:11] [INFO] Building visualizer...
[Genesis] [21:01:11] [INFO] Software rendering context detected. Shadows and plane reflection not supported. Beware rendering will be extremely slow.
--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ['policy']
	 critic :  ['policy']

In [41]:
print(obs, type(obs) is tuple)

tensor([[0.0000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],
       device='cuda:0') False
